# Week 6 — Spark Assignment
### Spark Architecture, Lazy Evaluation, Transformations, Filtering, Schema Handling & Optimized File Formats

**Dataset:** `DataSet/Superstore.csv` (Kaggle Sample Superstore dataset)


In [84]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col

# create the session
spark = SparkSession.builder \
    .appName("Spark_Data_Processing") \
    .getOrCreate()

spark


In [85]:
df = spark.read.csv(
    "DataSet/Superstore.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"'
)
df.show(5)


+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [86]:
# check the schema
df.printSchema()


root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [87]:
# original headers have spaces ("Product ID", "Order Date", etc.)
# renaming to snake_case so columns are easy to reference throughout the notebook
df = df.toDF(*[c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns])
df.printSchema()


root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)



## Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

### Answer

Apache Spark follows a master-worker architecture where different components work together to execute distributed data processing tasks.

### 1. Driver

The Driver is the main process of a Spark application. It creates the Spark Session, converts the user program into execution tasks, schedules jobs, and collects the final results from the executors.

**Responsibilities:**
- Creates Spark Session
- Converts code into jobs and stages
- Schedules tasks
- Coordinates execution
- Collects results

---

### 2. Cluster Manager

The Cluster Manager is responsible for managing the computing resources available in the cluster. It allocates CPU cores and memory to Spark applications and launches executors on worker nodes.

**Responsibilities:**
- Manages cluster resources
- Allocates memory and CPU
- Starts executors
- Monitors resource usage

---

### 3. Executor

Executors are worker processes that run on worker nodes. They execute the tasks assigned by the Driver and return the results after processing the data.

**Responsibilities:**
- Execute tasks
- Process partitions of data
- Store intermediate results
- Return results to the Driver

---


## Q2. How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

### Answer

Lazy Evaluation is one of the best techniques in Apache Spark. Instead of executing each transformation immediately, Spark records all transformations and waits until an action is called. During this time, Spark builds a Directed Acyclic Graph (DAG) of all operations.

When an action such as `show()`, `collect()`, or `count()` is executed, Spark analyzes the DAG and creates an optimized execution plan.

### Benefits of Lazy Evaluation

- Reduces unnecessary computations.
- Combines multiple transformations into a single optimized execution plan.
- Minimizes data movement across the cluster.
- Improves execution speed and resource utilization.

### Example

Suppose we write the following transformations:

```python
filtered_df = df.filter(col("category") == "Technology")
selected_df = filtered_df.select("product_id", "sales")
```

At this stage, Spark does **not** execute these operations.

Execution begins only when an action is performed:

```python
selected_df.show()
```

Spark then executes the entire workflow efficiently as a single optimized job instead of running each transformation separately.


## Q3. Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [88]:
# Read the CSV file
df = spark.read.csv(
    "DataSet/Superstore.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"'
)
df = df.toDF(*[
    c.strip().lower().replace(" ", "_").replace("-", "_")
    for c in df.columns
])

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|  customer_name|  segment|      country|           city|     state|postal_code|region|     product_id|       category|sub_category|        product_name|   sales|quantity|discount|  profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

### Answer

CSV and Parquet are two commonly used file formats in Apache Spark, but they differ in how data is stored and processed.

### CSV (Row-Based Storage)

CSV stores data row by row in plain text format. Each row contains all the values for a single record.

**Characteristics:**
- Row-based storage
- Human-readable text format
- Larger file size
- Slower read and write performance
- Does not store schema information
- Requires schema inference or manual schema definition

### Parquet (Columnar Storage)

Parquet stores data column by column in a binary format. Values from the same column are stored together, making it highly efficient for analytical workloads.

**Characteristics:**
- Columnar storage
- Compressed binary format
- Smaller file size
- Faster read performance
- Stores schema information
- Supports Predicate Pushdown and Column Pruning

---

### Conclusion

CSV is suitable for sharing and exchanging data because it is simple and readable. Parquet is the preferred format for Apache Spark applications because its columnar storage, compression, and built-in optimizations provide significantly better performance for large-scale data processing — which matters here since the full Superstore dataset has thousands of order lines.


## Q5. Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the `category` is 'Electronics'.

*Note: Superstore's real `category` values are `Furniture`, `Office Supplies`, and `Technology` — there is no `Electronics` category, so `Technology` is used as the equivalent. `sales` is used as the `price` field.*


In [89]:
df.filter(col("category") == "Technology") \
  .select("product_id", "sales") \
  .show()

+---------------+--------+
|     product_id|   sales|
+---------------+--------+
|TEC-PH-10002275| 907.152|
|TEC-PH-10002033| 911.424|
|TEC-PH-10001949|  213.48|
|TEC-AC-10003027|   90.57|
|TEC-PH-10004977|1097.544|
|TEC-PH-10000486| 371.168|
|TEC-PH-10004093| 147.168|
|TEC-AC-10000171|   45.98|
|TEC-AC-10002167|    45.0|
|TEC-PH-10003988|    21.8|
|TEC-PH-10002447| 1029.95|
|TEC-AC-10002167|    30.0|
|TEC-AC-10004633|   13.98|
|TEC-PH-10002726| 167.968|
|TEC-AC-10001998|   19.99|
|TEC-PH-10004093|  73.584|
|TEC-AC-10001767|  95.976|
|TEC-AC-10001552| 238.896|
|TEC-AC-10003499|  74.112|
|TEC-PH-10002844|  27.992|
+---------------+--------+
only showing top 20 rows


## Q6. Write the code to revise a DataFrame by renaming the column `old_name` to `new_name` and casting the `price` column from a String to a Double.

*We rename `product_name` to `new_name`, and cast `sales` (our `price` equivalent) to Double.*


In [90]:
revised_df = df.withColumnRenamed("product_name", "new_name") \
               .withColumn("sales", col("sales").cast("double"))

# Display the updated DataFrame
revised_df.show(5)

# Display the updated schema
revised_df.printSchema()


+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|  customer_name|  segment|      country|           city|     state|postal_code|region|     product_id|       category|sub_category|            new_name|   sales|quantity|discount|  profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

### Answer

Apache Spark provides fault tolerance through the **Lineage Graph**, also known as the **Directed Acyclic Graph (DAG)**. Instead of storing multiple copies of intermediate data, Spark records all the transformations applied to the data.

If a worker node or executor fails, Spark uses the Lineage Graph to identify the lost data partitions and recomputes only those partitions from the original data using the recorded transformations. This avoids reprocessing the entire dataset and improves the efficiency of recovery.

### Key Points

- Spark records every transformation in a Lineage Graph (DAG).
- The DAG maintains the sequence of operations performed on the data.
- If an executor fails, Spark identifies the lost partitions.
- Only the missing partitions are recomputed from the original data.
- This mechanism provides efficient fault tolerance without duplicating intermediate data.

---


## Q8. Write a query to filter a DataFrame `df_orders` for rows where the `status` is 'Completed' AND the `amount` is greater than 1000.

*Superstore has no order-status field, so `status` is derived here for the exercise: `Completed` when `profit` is positive, otherwise `Pending`. `amount` is represented by `sales`.*


In [91]:
df_orders = df.withColumn(
    "status",
    when(col("profit") > 0, "Completed").otherwise("Pending")
)

completed_orders = df_orders.filter(
    (col("status") == "Completed") &
    (expr("try_cast(sales as double)") > 1000)
)

completed_orders.show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+-----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+---------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|    customer_name|    segment|      country|         city|      state|postal_code| region|     product_id|       category|sub_category|        product_name|   sales|quantity|discount|   profit|   status|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+-----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+---------+
|    11|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710|  Brosina Hoffman|   Consumer|United States|  Los Angeles| California|      90032|   West|FUR-TA-100

## Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

### Answer

Predicate Pushdown is a performance optimization technique used by Apache Spark when reading Parquet files. Instead of loading the entire dataset into memory, Spark pushes the filtering condition down to the Parquet storage layer.

As a result, only the rows that satisfy the specified condition are read from the disk, while the remaining data is skipped. This significantly reduces disk I/O, memory usage, and query execution time.

### Key Points

- Predicate Pushdown is supported by Parquet files.
- Spark applies filter conditions before loading the data into memory.
- Only the required rows are read from the storage.
- It reduces disk I/O and memory consumption.
- It improves the overall performance of Spark applications.

---

### Example

Suppose we want to retrieve only the completed orders:

```python
completed_orders = df_orders.filter(col("status") == "Completed")
```

When the data is stored in **Parquet** format, Spark reads only the rows where the **status** is **Completed** instead of scanning the entire dataset.

---

### Benefits

- Faster query execution.
- Reduced memory usage.
- Less data transferred from disk.
- Better performance for large datasets.
- Efficient resource utilization.

### Conclusion

Predicate Pushdown is an important optimization feature of Parquet that allows Spark to read only the necessary data based on filter conditions. This minimizes the amount of data loaded into memory, resulting in faster and more efficient data processing.


## Q10. Write a code snippet to add a new column `final_price` which is the `base_price` multiplied by 1.18 (18% tax).

*Superstore has no separate `base_price` field, so `sales` is used as the base price before tax.*


In [92]:
updated_df = df.withColumn(
    "final_price",
    col("sales").cast("double") * 1.18
)

updated_df.select(
    "product_id",
    "sales",
    "final_price"
).show(5)

+---------------+--------+------------------+
|     product_id|   sales|       final_price|
+---------------+--------+------------------+
|FUR-BO-10001798|  261.96|309.11279999999994|
|FUR-CH-10000454|  731.94|          863.6892|
|OFF-LA-10000240|   14.62|           17.2516|
|FUR-TA-10000577|957.5775|        1129.94145|
|OFF-ST-10000760|  22.368|26.394239999999996|
+---------------+--------+------------------+
only showing top 5 rows


## Q11. What is the difference between Transformations and Actions? Provide two examples of each.

### Answer

In Apache Spark, operations are divided into **Transformations** and **Actions**.

### Transformations

Transformations are operations that create a new DataFrame or RDD from an existing one. They are **lazy**, meaning Spark does not execute them immediately. Instead, Spark records these operations and executes them only when an action is called.

**Examples:**
- `filter()`
- `select()`

Example:

```python
filtered_df = df.filter(col("category") == "Technology")
selected_df = filtered_df.select("product_id", "sales")
```

The above code defines the transformations, but Spark does **not** execute them immediately.

---

### Actions

Actions are operations that trigger the execution of all pending transformations. They either return a result to the Driver or write data to storage.

**Examples:**
- `show()`
- `count()`

Example:

```python
filtered_df.show()
filtered_df.count()
```

When an action is executed, Spark processes all the previous transformations and returns the required result.


## Q12. Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where `user_id` is null, and save the result as a CSV at "path/to/output".

*`user_id` is represented by `customer_id` in the Superstore data.*


In [93]:
# write the source data as parquet first, so we have a parquet file to read from
df.write.mode("overwrite").parquet("parquet_output")


In [94]:
# Load the Parquet file
parquet_df = spark.read.parquet("parquet_output")

# Display the first 5 records
parquet_df.show(5)

# Filter rows where customer_id (user_id) is not null
filtered_df = parquet_df.filter(col("customer_id").isNotNull())

# Display the filtered records
filtered_df.show()

# Save the filtered DataFrame as a CSV file
filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("csv_output")

print("Filtered data has been saved successfully as a CSV file.")


+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|  customer_name|  segment|      country|           city|     state|postal_code|region|     product_id|       category|sub_category|        product_name|   sales|quantity|discount|  profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

### Answer

In Apache Spark, **Client Mode** and **Cluster Mode** are two different deployment modes that determine where the **Driver Program** runs.

### Client Mode

In **Client Mode**, the Driver Program runs on the local machine from which the Spark application is submitted, while the Executors run on the cluster. The client machine must remain connected throughout the execution of the application. This mode is mainly used for development, testing, and debugging because it allows easy monitoring of the application.

### Cluster Mode

In **Cluster Mode**, the Driver Program runs inside the cluster on one of the worker nodes, and the Executors also run on the cluster. Once the application is submitted, the client can disconnect because the cluster manages the entire execution. This mode is mainly used for production environments and large-scale data processing.

---


## Q14. Write a query to filter a dataset for rows where the `region` is **'North'** OR the `priority` is **'High'**.

*Superstore's real `region` values are `South`, `West`, `East`, `Central` (no `North`), so `West` is used here. `priority` is derived from `ship_mode`: `Same Day` and `First Class` map to `High`.*


In [95]:
df_priority = df.withColumn(
    "priority",
    when(col("ship_mode").isin("Same Day", "First Class"), "High").otherwise("Low")
)

# Filter rows where region is 'West' OR priority is 'High'
filtered_df = df_priority.filter(
    (col("region") == "West") |
    (col("priority") == "High")
)

# Display the filtered records
filtered_df.show()


+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+--------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|     customer_name|  segment|      country|         city|     state|postal_code| region|     product_id|       category|sub_category|        product_name|   sales|quantity|discount|  profit|priority|
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+--------+
|     3|CA-2016-138688| 6/12/2016| 6/16/2016|  Second Class|   DV-13045|   Darrin Van Huff|Corporate|United States|  Los Angeles|California|      90036|   West|OFF-LA-10000240|Office S

## Q15. When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

### Answer

When working with large datasets in Apache Spark, it is safer to use **`.show(5)`** instead of **`.collect()`** because `.show(5)` displays only the first five rows of the DataFrame, while `.collect()` retrieves the entire dataset from all worker nodes and sends it to the Driver Program.

For very large datasets, using `.collect()` can consume a large amount of memory on the Driver, which may lead to slow performance or an **Out of Memory (OOM)** error. In contrast, `.show(5)` retrieves only a small sample of the data, making it much faster and memory-efficient.

---

### Example

```python
# Display only the first 5 records
df.show(5)
```

```python
# Retrieve the entire dataset to the Driver
data = df.collect()
```

---


In [96]:
# safe: only a small sample is returned
df.show(5)


+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|  customer_name|  segment|      country|           city|     state|postal_code|region|     product_id|       category|sub_category|        product_name|   sales|quantity|discount|  profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 